# Extended Analysis on LPIPS - randomized weights

In [1]:
import trsfile
import matplotlib.pyplot as plt
import numpy as np
from tqdm import trange, tqdm
import seaborn as sns
import bokeh.plotting as bk
import lpips
import librosa
import torchaudio
import torch
import math
from scipy.signal import find_peaks

from skimage.metrics import structural_similarity as ssim
import torch.nn.functional as F


from bokeh.io import output_notebook 
output_notebook()

Loading BokehJS ...

In [2]:
# Function to reset model weights
def reset_model(model):
    model.apply(reset_model_weights)
    return model

# Function to reset weights of individual layers
def reset_model_weights(layer):
    if hasattr(layer, 'reset_parameters'):
        layer.reset_parameters()
    elif hasattr(layer, 'children') and len(list(layer.children())) > 0:
        for child in layer.children():
            reset_model_weights(child)

In [3]:
## LPIPS model
device = "cuda" if torch.cuda.is_available() else "cpu" 
lpips_model = lpips.LPIPS(net='alex').to(device)
# Save original weights
original_weights = {name: param.clone() for name, param in lpips_model.named_parameters()}

ablation_model = reset_model(lpips_model)

ablation_model.eval()
ablation_model.to(device)

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/Users/lizzy/Documents/Education/PhD/Projects/LPIPS-OTA3/lpips-ota3/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/lizzy/Documents/Education/PhD/Projects/LPIPS-OTA3/lpips-ota3/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /Users/lizzy/Documents/Education/PhD/Projects/LPIPS-OTA3/lpips-ota3/.venv/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth


LPIPS(
  (scaling_layer): ScalingLayer()
  (net): alexnet(
    (slice1): Sequential(
      (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
      (1): ReLU(inplace=True)
    )
    (slice2): Sequential(
      (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
      (4): ReLU(inplace=True)
    )
    (slice3): Sequential(
      (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): ReLU(inplace=True)
    )
    (slice4): Sequential(
      (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (9): ReLU(inplace=True)
    )
    (slice5): Sequential(
      (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
    )
  )
  (lin0): NetLinLayer(
    (model): Sequential(
      (0): D

In [4]:
# Compare weights after resetting
for name, param in ablation_model.named_parameters():
    if not torch.equal(original_weights[name], param):
        print(f"Weights for layer '{name}' have been successfully reset.")
    else:
        print(f"Weights for layer '{name}' have NOT been reset.")

Weights for layer 'net.slice1.0.weight' have been successfully reset.
Weights for layer 'net.slice1.0.bias' have been successfully reset.
Weights for layer 'net.slice2.3.weight' have been successfully reset.
Weights for layer 'net.slice2.3.bias' have been successfully reset.
Weights for layer 'net.slice3.6.weight' have been successfully reset.
Weights for layer 'net.slice3.6.bias' have been successfully reset.
Weights for layer 'net.slice4.8.weight' have been successfully reset.
Weights for layer 'net.slice4.8.bias' have been successfully reset.
Weights for layer 'net.slice5.10.weight' have been successfully reset.
Weights for layer 'net.slice5.10.bias' have been successfully reset.
Weights for layer 'lin0.model.1.weight' have been successfully reset.
Weights for layer 'lin1.model.1.weight' have been successfully reset.
Weights for layer 'lin2.model.1.weight' have been successfully reset.
Weights for layer 'lin3.model.1.weight' have been successfully reset.
Weights for layer 'lin4.mode

In [5]:
def calc_spectrogram(trace, n_fft = 512): #NOTE: it is very sensitive for this value
    trace_tensor = torch.tensor(trace, dtype=torch.float32)
    spec = torchaudio.transforms.Spectrogram(n_fft=n_fft)(trace_tensor)
    spec_db = librosa.power_to_db(spec.numpy(), ref=np.max)

    # Normalize to [0,1]
    spec_db = (spec_db - spec_db.min()) / (spec_db.max() - spec_db.min() + 1e-8)

    return spec_db

def spec_to_tensor(spec_db, target_size):
    # Convert to 3-channel float32 tensor for LPIPS: (C, H, W)
    spec_tensor = torch.tensor(spec_db).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
    # Resize to target_size
    spec_resized = F.interpolate(spec_tensor, size=target_size, mode='bilinear', align_corners=False)

    return spec_resized

def compute_distance_lpips(target_trace, template_trace):
    tensor_size = (224, 224)  # LPIPS expects 224x224 input
    
    target_spec  = calc_spectrogram(target_trace, 512)
    template_spec = calc_spectrogram(template_trace, 512)

    target_tensor = spec_to_tensor(target_spec, tensor_size).to(device)
    template_tensor = spec_to_tensor(template_spec, tensor_size).to(device)

    d = lpips_model.forward(target_tensor,template_tensor)
    return d.detach().cpu().numpy().ravel()[0]

In [6]:
def plot_similarity(similarity_array0, similarity_array1, scalars, left, right, similarity_function):
        plt.figure(figsize=(10,5))
        plt.plot(similarity_array0, label = "bit 0")
        plt.plot(similarity_array1, label = "bit 1")
        plt.axvspan(left, right, alpha=0.2)
        plt.title("Similarity for template for {} on {}".format(scalars, similarity_function.__name__))
        plt.xlabel("Timesamples")
        plt.ylabel("Delta power consumption")
        plt.legend()
        #plt.savefig("./ota1_bitpos{}_{}.png".format(iteration, similarity_function.__name__), bbox_inches='tight')
        plt.show()

In [7]:
# Function to cut off the "0 part" in template traces
def slice_trace(trace):
    idx = np.where(trace == 0)[0][0] # Get the first index where the power consumption = 0 [note, might not work for all datasets]
    new_trace = trace[:idx]
    return new_trace

def pattern_match(target_trace, template_trace, comparison_function, poi_start, poi_end, step=1):
    comparison_array = []
    window = poi_end - poi_start
    template_trace_sliced = template_trace[poi_start:poi_end]
    for timesample in trange(0, len(target_trace) - window, step):
        comparison = comparison_function(target_trace[timesample:timesample + window], template_trace_sliced)
        comparison_array.append(comparison)
    return np.array(comparison_array)

def determine_keybit(comparison_array0, comparison_array1, sign):
    # Get the highest peak of both, and take that timesample as comparison point
    if sign == "max":
        peak0 = np.max(comparison_array0)
        peak1 = np.max(comparison_array1)
        keybit_guess = np.argmax([peak0, peak1]) 
        print("Score bit 0: ", peak0)
        print("Score bit 1: ", peak1)
        timesample = np.argmax(comparison_array0) if keybit_guess == 0 else np.argmax(comparison_array1)
    elif sign == "min":
        peak0 = np.min(comparison_array0)
        peak1 = np.min(comparison_array1)
        keybit_guess = np.argmin([peak0, peak1]) 
        print("Score bit 0: ", peak0)
        print("Score bit 1: ", peak1)
        timesample = np.argmin(comparison_array0) if keybit_guess == 0 else np.argmin(comparison_array1)
    
    return int(keybit_guess), timesample

## OTA on NaCl

In [26]:
## Open the dataset
with trsfile.open('../../datasets/ota1/attack + affine_templates.trs', 'r') as traces:
    # Show all headers

    for header, value in traces.get_headers().items():
        print(header, '=', value)

        if str(header) == "Header.NUMBER_TRACES":
            nr_traces = value
        elif str(header) == "Header.NUMBER_SAMPLES":
            nr_samples = value
    print()

    trace_array = np.zeros((nr_traces, nr_samples))

    # Iterate over the first 11 traces
    for i, trace in enumerate(traces):
        print('Trace {0:d} contains {1:d} samples'.format(i, len(trace)))
        print('  - minimum value in trace: {0:f}'.format(min(trace)))
        print('  - maximum value in trace: {0:f}'.format(max(trace)))
        trace_array[i, :] = trace



sample_rate = 125e6 

# Split the target trace from the template traces
target_trace = trace_array[0, :]
raw_template_traces = trace_array[1:, :]

print(target_trace.shape) 

# Cut off all 0 for each template trace
template_traces = np.zeros((10, np.where(raw_template_traces[0] == 0)[0][0]))

for i, trace in enumerate(raw_template_traces):
    new_trace = slice_trace(trace)
    template_traces[i] = new_trace

print(template_traces.shape)

Header.NUMBER_TRACES = 11
Header.NUMBER_SAMPLES = 819170
Header.SAMPLE_CODING = SampleCoding.FLOAT
Header.LENGTH_DATA = 98
Header.TRACE_TITLE = trace
Header.LABEL_X = sec
Header.LABEL_Y = Volt
Header.SCALE_X = 2.4999948777804093e-07
Header.SCALE_Y = 0.015748031437397003
Header.TRACE_BLOCK = None

Trace 0 contains 819170 samples
  - minimum value in trace: -124.225807
  - maximum value in trace: -97.741936
Trace 1 contains 819170 samples
  - minimum value in trace: -124.580643
  - maximum value in trace: 0.000000
Trace 2 contains 819170 samples
  - minimum value in trace: -124.290321
  - maximum value in trace: 0.000000
Trace 3 contains 819170 samples
  - minimum value in trace: -124.193550
  - maximum value in trace: 0.000000
Trace 4 contains 819170 samples
  - minimum value in trace: -124.258064
  - maximum value in trace: 0.000000
Trace 5 contains 819170 samples
  - minimum value in trace: -124.483871
  - maximum value in trace: 0.000000
Trace 6 contains 819170 samples
  - minimum va

In [27]:
# Do the sequential key bit recovery
keybits = 5
step_size = 1

scalars = ["2P vs 3P", "4P vs 5P", "8P vs 9P", "18P vs 19P", "38P vs 39P"]

poi_start = 2024  # This window determined by pattern identification, such as e.g. autocorrelation (used in the original OTA paper)
poi_end = 8622
window = poi_end - poi_start

recovered_key = np.ones(keybits + 1)  # Initialize with ones to set the first bit for each method to 1

valleys = [0, 135000, 312000, 490000, 670000, 810000] # using SPA to speed up computation time


for iteration, keybit_to_guess in enumerate(range(0, keybits*2, 2)):
    # Determine input points based on previous key bits
    # [It is an online attack, but as the dataset is already there, assume that new input points are calculated here]
    
    # Collect traces
    # [It is an online attack, but as the dataset is already there, assume that new input points are put in the code & traces are collected]

    # Create templates from the collected traces
    template0 = template_traces[keybit_to_guess, :]
    template1 = template_traces[keybit_to_guess+1, :]

    # Select current window to attack based on SPA. Only done to speed up computation time
    start_window = valleys[iteration]
    end_window = valleys[iteration + 1]

    # Slice target trace based on the window where we attack
    target_slice = target_trace[start_window:end_window]

    similarity_array0 = pattern_match(target_slice, template0, compute_distance_lpips, poi_start, poi_end, step=step_size)
    similarity_array1 = pattern_match(target_slice, template1, compute_distance_lpips, poi_start, poi_end, step=step_size)

    np.savez("./analysis/analysis_ota1_bitpos{}_{}.npz".format(iteration, compute_distance_lpips.__name__), comparison_0 = similarity_array0, comparison_1 = similarity_array1)

    best_keyguess, timesample  = determine_keybit(similarity_array0, similarity_array1, sign="min")

    #plot_similarity(similarity_array0, similarity_array1, scalars[iteration], timesample-5000, timesample+5000, compute_distance_lpips)

    recovered_key[iteration+1] = best_keyguess
    print("Recovered keybit: ", best_keyguess)

print("The correct key: 100110")
print(recovered_key)


100%|██████████| 128402/128402 [19:48<00:00, 108.01it/s]


Score bit 0:  -3.610109e-05
Score bit 1:  -4.9840324e-05
Recovered keybit:  1


100%|██████████| 170402/170402 [27:04<00:00, 104.86it/s]


Score bit 0:  -4.5743116e-05
Score bit 1:  -4.9781833e-05
Recovered keybit:  1


100%|██████████| 171402/171402 [27:03<00:00, 105.55it/s]


Score bit 0:  -3.8982118e-05
Score bit 1:  -4.7214704e-05
Recovered keybit:  1


100%|██████████| 173402/173402 [27:42<00:00, 104.28it/s]


Score bit 0:  -3.5235145e-05
Score bit 1:  -3.3127195e-05
Recovered keybit:  0


100%|██████████| 133402/133402 [21:09<00:00, 105.07it/s]

Score bit 0:  -4.2996588e-05
Score bit 1:  -5.5419645e-05
Recovered keybit:  1
The correct key: 100110
[1. 1. 1. 1. 0. 1.]


## OTA on mbedTLS

In [10]:
poi_start = 1079000
poi_end = 1098000
window = poi_end - poi_start
step_size = 10

## Scalar  = 2 => 10 in binary
single_target_trace_sc2 = np.array(trsfile.open("../../datasets/ota2/P1_scalar2_unprotected_ec_scalmul.trs", 'r')[0], dtype=np.float64)
single_template_2P_sc2 = np.array(trsfile.open('../../datasets/ota2/2P1_scalar2_unprotected_ec_scalmul.trs', 'r')[0], dtype=np.float64)
single_template_3P_sc2 = np.array(trsfile.open('../../datasets/ota2/3P1_scalar2_unprotected_ec_scalmul.trs', 'r')[0], dtype=np.float64)

## Scalar  = 3 => 11 in binary
single_target_trace_sc3 = np.array(trsfile.open("../../datasets/ota2/P1_scalar3_unprotected_ec_scalmul.trs", 'r')[0], dtype = np.float64)
single_template_2P_sc3 = np.array(trsfile.open('../../datasets/ota2/2P1_scalar3_unprotected_ec_scalmul.trs', 'r')[0], dtype = np.float64)
single_template_3P_sc3 = np.array(trsfile.open('../../datasets/ota2/3P1_scalar3_unprotected_ec_scalmul.trs', 'r')[0], dtype = np.float64)


single_trace_leakage_sc2 = [single_target_trace_sc2, single_template_2P_sc2, single_template_3P_sc2]
single_trace_leakage_sc3 = [single_target_trace_sc3, single_template_2P_sc3, single_template_3P_sc3]

In [11]:
for correct_keybit, single_trace_leakage in enumerate([single_trace_leakage_sc2, single_trace_leakage_sc3]):
    comparison_0 = pattern_match(single_trace_leakage[0], single_trace_leakage[1], compute_distance_lpips, poi_start, poi_end, step=step_size)
    comparison_1 = pattern_match(single_trace_leakage[0], single_trace_leakage[2], compute_distance_lpips, poi_start, poi_end, step=step_size)

    #plot_similarity(comparison_0, comparison_1, correct_keybit, similarity_function=compute_distance_lpips, leakage = "horizontal")
    keybit_guess, timesample = determine_keybit(comparison_0, comparison_1, sign="min") 
    
    print("The correct keybit is: 1", correct_keybit)
    print("The found keybit using {} is: 1{}".format(compute_distance_lpips.__name__, keybit_guess))

    np.savez("./analysis/analysis_ota2_sc1{}_{}".format(correct_keybit, compute_distance_lpips.__name__),  comparison_0 = comparison_0, comparison_1 = comparison_1)


  0%|          | 0/2498100 [00:00<?, ?it/s]/home/lizzy/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 2498100/2498100 [5:26:43<00:00, 127.43it/s]  


Score bit 0:  7.74772e-05
Score bit 1:  7.563559e-05
The correct keybit is: 1 0
The found keybit using compute_distance_lpips is: 11


100%|██████████| 2498100/2498100 [5:49:50<00:00, 119.01it/s]  


Score bit 0:  7.348689e-05
Score bit 1:  8.3973726e-05
The correct keybit is: 1 1
The found keybit using compute_distance_lpips is: 10


## OTA on sca25519

In [8]:
traces = np.load("../ecc-scalarmult-montgomery.npz")["traces"]
target_trace = traces[0]
template_traces = traces[1:]
nr_samples = len(target_trace)
print(template_traces)

print(nr_samples)

[[-0.06461184 -0.00504153 -0.00015872 ...  0.07186276  0.0318237
   0.00740964]
 [-0.06878849 -0.00238224 -0.00335881 ...  0.07403377  0.03008846
   0.00250057]
 [-0.06557604 -0.00185533 -0.00063463 ...  0.0740724   0.03647474
   0.01303724]
 ...
 [-0.0653283  -0.00795526  0.00229865 ...  0.07139045  0.0308631
   0.00400763]
 [-0.06532622 -0.00184965 -0.00258208 ...  0.07456636  0.03281832
   0.00864839]
 [-0.0557879  -0.00525079 -0.01257501 ...  0.07922186  0.03552069
   0.00573554]]
300000


In [9]:
def compute_correlation(target_trace, template_trace):
    corr = np.corrcoef(target_trace, template_trace)[0,1] # Take the non-identity part of the correlation matrix. Note that [0,1] = [1,0]
    return np.abs(corr)

def get_window(any_template, iteration, step_size=1, height=0.7, distance=20000, prominence=0.15):
    #signal = pattern_match(target_trace, any_template, compute_correlation, poi_start, poi_end)
    peaks, _ = find_peaks(any_template, height=height, distance=distance, prominence=prominence)
    left = peaks[iteration+1] - (distance // 2)
    right = peaks[iteration+1] + (distance // 2)
    
    return left//step_size, right//step_size

In [10]:
# Do the sequential key bit recovery

keybits = 6
scalar = 1
recovered_key = np.ones(keybits + 2)  # Initialize with ones to set the first bit for each method to 1
recovered_key[0] = 0  # Set the MSB to 1 for all methods

poi_start = 10000
poi_end = 37900
step_size = 1


# Perform pattern matching with templates to target trace based on Correlation
scalar = 1
for iteration, keybit_to_guess in enumerate(range(0, keybits*2, 2)):

    # Determine input points based on previous key bits
    # [It is an online attack, but as the dataset is already there, assume that new input points are calculated here]
    
    # Collect traces
    # [It is an online attack, but as the dataset is already there, assume that new input points are put in the code & traces are collected]

    # Create templates from the collected traces
    template0_targetbit = template_traces[2*iteration, :]
    template1_targetbit = template_traces[2*iteration + 1, :]
    templates_targetbit = [template0_targetbit, template1_targetbit]

    # Find peaks automatically
    corr_array = np.abs(pattern_match(target_trace, templates_targetbit[0], compute_correlation, poi_start, poi_end, step=1))
    window_start, window_end = get_window(corr_array, iteration, step_size=step_size, height=0.7, distance=20000, prominence=0.15)

    comparison_arrays = []
    for template in templates_targetbit:
        comparison_array = pattern_match(target_trace, template, compute_distance_lpips, poi_start, poi_end, step=step_size)
        comparison_arrays.append(comparison_array)

    np.savez("./results/analysis_ota3_bitpos{}_{}.npz".format(iteration, compute_distance_lpips.__name__), comparison_0 = comparison_arrays[0], comparison_1 = comparison_arrays[1])

    #plot_similarity(comparison_arrays, scalar, window_start, window_end, compute_distance_lpips)
    comparison_array0 = comparison_arrays[0][window_start:window_end]
    comparison_array1 = comparison_arrays[1][window_start:window_end]

    best_keyguess, timesample = determine_keybit(comparison_array0, comparison_array1, "min")
    
    recovered_key[iteration+1] = best_keyguess
    print("Best key bit for method {}: {}".format(compute_distance_lpips.__name__, best_keyguess))

print("The correct key: 01101010")

print(recovered_key)

    

  0%|          | 0/272100 [00:00<?, ?it/s]

100%|██████████| 272100/272100 [45:56<00:00, 98.72it/s] 


Score bit 0:  -0.00017261006
Score bit 1:  -0.00017331149
Best key bit for method compute_distance_lpips: 1


100%|██████████| 272100/272100 [46:00<00:00, 98.57it/s] 


Score bit 0:  -0.00016737862
Score bit 1:  -0.00016848385
Best key bit for method compute_distance_lpips: 1


...

Score bit 0:  -0.00017271184
Score bit 1:  -0.00017565506
Best key bit for method compute_distance_lpips: 1


100%|██████████| 272100/272100 [47:15<00:00, 95.97it/s] 


Score bit 0:  -0.00017674727
Score bit 1:  -0.00017573939
Best key bit for method compute_distance_lpips: 0


100%|██████████| 272100/272100 [48:33<00:00, 93.38it/s]

Score bit 0:  -0.00017199115
Score bit 1:  -0.00017552663
Best key bit for method compute_distance_lpips: 1
The correct key: 01101010
[0. 1. 1. 0. 1. 0. 1. 1.]
